# Song Extractor

Extract song lyrics from photos/PDFs using Claude's vision API, then bulk upload to Supabase.

## Setup
1. `pip install -r requirements.txt`
2. Set your `ANTHROPIC_API_KEY` environment variable (or paste it below)
3. Put your song images/PDFs in a folder
4. Run the cells below

pip 25.1 from /opt/homebrew/Caskroom/miniconda/base/envs/basic/lib/python3.12/site-packages/pip (python 3.12)


In [ ]:
import anthropic
import base64
import json
import os
from pathlib import Path
from datetime import datetime, timezone

In [ ]:
# --- CONFIG ---

# Your Anthropic API key (or set ANTHROPIC_API_KEY env var)
API_KEY = os.environ.get("ANTHROPIC_API_KEY", "sk-ant-...")

# Category for the songs being extracted
CATEGORY = "special"  # change to: "youth-camp", "special", "yc-chorus", etc.

# Starting song number (will auto-increment per image)
START_NUMBER = 1

# Folder containing your images or PDF
INPUT_PATH = "./song_images"  # folder of images OR a single PDF path like "./songs.pdf"

# Output file
OUTPUT_FILE = "songs.json"

In [ ]:
client = anthropic.Anthropic(api_key=API_KEY)

SYSTEM_PROMPT = """You extract song lyrics from images into structured JSON.

Rules:
- Output ONLY valid JSON, no markdown fences, no explanation
- If the image contains multiple songs, return a JSON array of song objects
- If the image contains one song, still return a JSON array with one object
- Preserve the original language (Hindi, English, or mixed)
- Identify chorus/refrain sections and mark them with is_chorus: true
- Label verses as "Verse 1", "Verse 2", etc.
- Label chorus as "Chorus"
- If a song number is visible in the image, use it
- Keep line breaks within stanzas as \n in the text field
- If the title is visible, use it. If not, use the first line as the title.

Output schema per song:
{
  "number": <integer>,
  "title": "<song title>",
  "stanzas": [
    { "label": "Verse 1", "text": "line1\nline2\nline3", "is_chorus": false },
    { "label": "Chorus", "text": "line1\nline2", "is_chorus": true },
    ...
  ]
}"""

In [ ]:
def load_image_as_base64(path: str) -> tuple[str, str]:
    """Returns (base64_data, media_type) for an image file."""
    ext = Path(path).suffix.lower()
    media_types = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }
    media_type = media_types.get(ext, "image/jpeg")
    with open(path, "rb") as f:
        data = base64.standard_b64encode(f.read()).decode("utf-8")
    return data, media_type


def load_pdf_as_base64(path: str) -> str:
    """Returns base64 data for a PDF file."""
    with open(path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode("utf-8")


def extract_songs_from_content(content_blocks: list, song_number_hint: int | None = None) -> list[dict]:
    """Send content blocks to Claude and parse the JSON response."""
    user_msg = "Extract all song lyrics from this image into the JSON format described."
    if song_number_hint is not None:
        user_msg += f" If no song number is visible, use {song_number_hint} as the number."

    content_blocks.append({"type": "text", "text": user_msg})

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=4096,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": content_blocks}],
    )

    raw = response.content[0].text.strip()
    # Strip markdown fences if present
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1]
        raw = raw.rsplit("```", 1)[0]

    songs = json.loads(raw)
    if isinstance(songs, dict):
        songs = [songs]
    return songs

In [ ]:
all_songs = []
input_path = Path(INPUT_PATH)
current_number = START_NUMBER

if input_path.is_file() and input_path.suffix.lower() == ".pdf":
    # --- PDF mode: send the whole PDF at once ---
    print(f"Processing PDF: {input_path}")
    pdf_data = load_pdf_as_base64(str(input_path))
    content = [
        {
            "type": "document",
            "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_data},
        }
    ]
    songs = extract_songs_from_content(content, song_number_hint=current_number)
    for song in songs:
        song["category"] = CATEGORY
        all_songs.append(song)
    print(f"  Extracted {len(songs)} song(s)")

elif input_path.is_dir():
    # --- Image folder mode: one image at a time ---
    image_files = sorted(
        [f for f in input_path.iterdir() if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".gif", ".webp")]
    )
    print(f"Found {len(image_files)} images in {input_path}")

    for img_path in image_files:
        print(f"Processing: {img_path.name} (hint number: {current_number})")
        img_data, media_type = load_image_as_base64(str(img_path))
        content = [
            {
                "type": "image",
                "source": {"type": "base64", "media_type": media_type, "data": img_data},
            }
        ]
        songs = extract_songs_from_content(content, song_number_hint=current_number)
        for song in songs:
            song["category"] = CATEGORY
            all_songs.append(song)
            current_number = max(current_number, song.get("number", current_number)) + 1
        print(f"  Extracted {len(songs)} song(s)")

else:
    print(f"ERROR: {INPUT_PATH} is not a valid folder or PDF file")

print(f"\nTotal songs extracted: {len(all_songs)}")

In [ ]:
# Preview the extracted songs
for s in all_songs:
    print(f"#{s['number']} - {s['title']} ({len(s['stanzas'])} stanzas)")

In [ ]:
# --- REVIEW & EDIT ---
# Inspect / fix any song before saving. Example:
# all_songs[0]["title"] = "Corrected Title"
# all_songs[0]["stanzas"][1]["is_chorus"] = True

# Print full JSON for review
print(json.dumps(all_songs, indent=2, ensure_ascii=False))

In [ ]:
# Save to songs.json
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_songs, f, indent=2, ensure_ascii=False)

print(f"Saved {len(all_songs)} songs to {OUTPUT_FILE}")

In [ ]:
# --- UPLOAD TO SUPABASE ---
# Set these env vars or paste values directly
SUPABASE_URL = os.environ.get("SUPABASE_URL", "https://your-project.supabase.co")
SUPABASE_SERVICE_KEY = os.environ.get("SUPABASE_SERVICE_KEY", "your-service-role-key")

from supabase import create_client

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)
now = datetime.now(timezone.utc).isoformat()

rows = [
    {
        "category": song["category"],
        "number": song["number"],
        "title": song["title"],
        "stanzas": song["stanzas"],
        "updated_at": now,
    }
    for song in all_songs
]

result = supabase.table("songs").upsert(rows, on_conflict="category,number").execute()
print(f"Upserted {len(result.data)} songs to Supabase")